# Introduction to DuckDB

## What is DuckDB?

DuckDB is an in-process SQL OLAP database management system. It's designed for analytical workloads and is optimized for fast queries on large datasets. Unlike traditional databases that run as separate servers, DuckDB runs embedded within your application.

## Key Features

- **Fast**: Optimized for analytical queries
- **Embedded**: No server setup required
- **SQL**: Standard SQL syntax
- **Python Integration**: Easy to use with pandas and other Python libraries
- **File-based**: Can work with CSV, Parquet, and other formats directly

## Why DuckDB for Analytics?

- Perfect for data exploration and prototyping
- Great for local development with dbt
- Can handle datasets that fit in memory
- Excellent performance for complex analytical queries

## Setting Up DuckDB

First, let's install and import DuckDB.

In [1]:
# Install DuckDB (if not already installed)
# !pip install duckdb

# Import required libraries
import duckdb
import polars as pl
import os

print("DuckDB version:", duckdb.__version__)
print("Polars version:", pl.__version__)

DuckDB version: 0.10.2
Polars version: 1.39.3


## Connecting to Our Database

Our dbt project uses a DuckDB database file. Let's connect to it and explore the structure.

In [2]:
# Connect to the DuckDB database
# The database file is created by create_db.py (loads Parquet files from data/)
db_path = "../my_database.duckdb"

if os.path.exists(db_path):
    con = duckdb.connect(db_path, read_only=True)
    print("Connected to DuckDB database successfully!")
else:
    raise FileNotFoundError(
        f"Database file not found at {db_path}. "
        "Run `python create_db.py` from the project root first."
    )

Connected to DuckDB database successfully!


## Exploring the Database Schema

Let's see what tables are available in our database.

In [3]:
# List all tables in the database
tables_query = """
SELECT table_name 
FROM information_schema.tables 
WHERE table_schema = 'main'
ORDER BY table_name;
"""

tables = con.execute(tables_query).fetchall()
print("Tables in the database:")
for table in tables:
    print(f"- {table[0]}")

Tables in the database:
- categories
- customer_addresses
- customers
- inventory
- marketing_campaigns
- order_items
- orders
- payments
- products
- returns
- reviews
- segments
- shipping
- stg_customers
- suppliers
- website_sessions


## Basic Queries

Let's start with some basic SQL queries to explore our data.

In [4]:
# Query the customers table
customers_query = """
SELECT * 
FROM customers 
LIMIT 5;
"""

customers_df = con.execute(customers_query).fetchdf()
print("Sample customers:")
customers_df

Sample customers:


,customer_id,first_name,last_name,email,phone,age,gender,country,state,city,registration_date,customer_segment,total_orders,total_spent,is_active
0,1,First_1,Last_1,customer_1@example.com,+1-804-532-1520,19,F,France,None,City_29,2025-05-08,Bronze,1,366.58,True
1,2,First_2,Last_2,customer_2@example.com,+1-484-928-1106,50,M,Netherlands,None,City_26,2024-09-02,Silver,5,829.04,True
2,3,First_3,Last_3,customer_3@example.com,+1-299-467-6635,62,F,Australia,None,City_36,2025-01-30,Bronze,20,2703.53,True
3,4,First_4,Last_4,customer_4@example.com,+1-833-982-6925,64,F,Netherlands,None,City_16,2025-09-25,Platinum,2,451.28,True
4,5,First_5,Last_5,customer_5@example.com,+1-573-266-7065,20,M,Canada,None,City_11,2025-08-02,Gold,8,1206.44,True


In [5]:
# Query the products table
products_query = """
SELECT * 
FROM products 
LIMIT 5;
"""

products_df = con.execute(products_query).fetchdf()
print("Sample products:")
products_df

Sample products:


,product_id,product_name,category_id,supplier_id,price,cost,weight_kg,dimensions_cm,description,sku,barcode,status,created_date,is_featured
0,1,Desk Chair Desk Chair 530,26,27,205.73,130.90,34.28,183x182x84,High-quality desk chair desk chair 530 perfect...,SKU312876,761920122355,discontinued,2025-09-20,False
1,2,Dumbbells Dumbbells 131,46,75,96.77,55.05,9.17,36x42x16,High-quality dumbbells dumbbells 131 perfect f...,SKU266240,744765006139,active,2025-10-13,False
2,3,Laptop Laptop 487,1,82,990.60,409.23,46.09,127x6x187,High-quality laptop laptop 487 perfect for eve...,SKU152367,250328968993,active,2024-10-18,False
3,4,Blender Blender 657,27,52,113.19,53.66,22.45,120x84x27,High-quality blender blender 657 perfect for e...,SKU540181,322548952013,active,2024-11-23,False
4,5,Coffee Maker Basic 133,27,41,53.46,21.26,23.57,14x155x194,High-quality coffee maker basic 133 perfect fo...,SKU504877,400540557711,active,2024-05-23,False


In [6]:
# Query the orders table
orders_query = """
SELECT * 
FROM orders 
LIMIT 5;
"""

orders_df = con.execute(orders_query).fetchdf()
print("Sample orders:")
orders_df

Sample orders:


,order_id,customer_id,order_date,status,subtotal,tax_amount,shipping_cost,discount_amount,total_amount,currency,payment_method,shipping_address_id,billing_address_id,coupon_code
0,1,373,2025-06-28 22:49:46,completed,176.77,16.90,0.00,0.0,193.67,USD,google_pay,1807,1016,None
1,2,972,2024-03-02 22:49:46,completed,31.35,1.57,11.62,0.0,44.54,USD,bank_transfer,755,1934,None
2,3,1387,2024-06-29 22:49:46,completed,57.63,3.60,0.00,0.0,61.23,USD,paypal,100,696,None
3,4,1156,2024-05-30 22:49:46,completed,65.45,3.62,0.00,0.0,69.07,USD,credit_card,288,515,None
4,5,710,2025-02-18 22:49:46,shipped,51.18,4.85,0.00,0.0,56.03,USD,apple_pay,1925,345,None


## Filtering and Sorting

Let's practice filtering and sorting data.

In [7]:
# Find customers with gmail addresses
location_query = """
SELECT customer_id, first_name, last_name, email
FROM customers
WHERE email LIKE '%gmail.com'
ORDER BY last_name, first_name
LIMIT 10;
"""

gmail_customers = con.execute(location_query).fetchdf()
print("Customers with Gmail addresses:")
gmail_customers

Customers with Gmail addresses:


,customer_id,first_name,last_name,email


In [8]:
# Find expensive products
expensive_products_query = """
SELECT product_id, product_name, price
FROM products
WHERE price > 50
ORDER BY price DESC;
"""

expensive_products = con.execute(expensive_products_query).fetchdf()
print("Expensive products (price > $50):")
expensive_products

Expensive products (price > $50):


,product_id,product_name,price
0,478,Sofa Sofa 862,1910.20
1,257,Sofa Sofa 235,1809.17
2,146,Sofa Sofa 792,1800.84
3,952,Sofa Sofa 751,1769.21
4,1024,Sofa Sofa 899,1767.55
...,...,...,...
747,51,Coffee Maker Coffee Maker 271,50.94
748,261,Paint Set Paint Set 682,50.71
749,421,Coffee Maker Coffee Maker 313,50.65
750,445,Dumbbells Pro 574,50.40


## Joins

Now let's practice joining tables to get more meaningful insights.

In [9]:
# Join orders with customers
orders_customers_query = """
SELECT
    o.order_id,
    o.order_date,
    o.total_amount,
    c.first_name,
    c.last_name,
    c.email
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
ORDER BY o.order_date DESC
LIMIT 10;
"""

orders_with_customers = con.execute(orders_customers_query).fetchdf()
print("Recent orders with customer information:")
orders_with_customers

Recent orders with customer information:


,order_id,order_date,total_amount,first_name,last_name,email
0,2194,2025-12-22 22:49:46,54.49,First_1103,Last_1103,customer_1103@example.com
1,685,2025-12-21 22:49:46,0.00,First_331,Last_331,customer_331@example.com
2,1740,2025-12-21 22:49:46,92.15,First_592,Last_592,customer_592@example.com
3,1164,2025-12-21 22:49:46,115.20,First_1377,Last_1377,customer_1377@example.com
4,585,2025-12-21 22:49:46,0.00,First_25,Last_25,customer_25@example.com
5,1720,2025-12-21 22:49:46,129.34,First_660,Last_660,customer_660@example.com
6,73,2025-12-21 22:49:46,0.00,First_1067,Last_1067,customer_1067@example.com
7,1387,2025-12-20 22:49:46,113.73,First_150,Last_150,customer_150@example.com
8,1952,2025-12-20 22:49:46,89.63,First_843,Last_843,customer_843@example.com
9,815,2025-12-19 22:49:46,26.64,First_1322,Last_1322,customer_1322@example.com


In [10]:
# Aggregate orders per customer
order_summary_query = """
SELECT
    customer_id,
    COUNT(*) as total_orders,
    SUM(total_amount) as total_spent,
    AVG(total_amount) as avg_order_value,
    MAX(order_date) as last_order_date
FROM orders
GROUP BY customer_id
ORDER BY total_spent DESC
LIMIT 10;
"""

order_summary = con.execute(order_summary_query).fetchdf()
print("Top customers by total order amount:")
order_summary

Top customers by total order amount:


,customer_id,total_orders,total_spent,avg_order_value,last_order_date
0,124,4,826.97,206.742500,2025-04-26 22:49:46
1,545,5,804.75,160.950000,2025-12-13 22:49:46
2,492,7,795.93,113.704286,2025-09-01 22:49:46
3,748,6,770.50,128.416667,2025-07-28 22:49:46
4,295,6,765.24,127.540000,2025-12-03 22:49:46
5,1367,6,741.11,123.518333,2025-09-10 22:49:46
6,323,6,709.56,118.260000,2025-04-22 22:49:46
7,1148,6,699.10,116.516667,2025-08-03 22:49:46
8,1357,4,688.36,172.090000,2025-05-07 22:49:46
9,1379,7,685.81,97.972857,2025-05-01 22:49:46


## Aggregations and Analytics

Let's create some analytical queries with aggregations.

In [11]:
# Monthly sales analysis
# Note: order_date is stored as VARCHAR in the raw data, so we cast it to DATE
monthly_sales_query = """
SELECT
    DATE_TRUNC('month', order_date::DATE) as month,
    COUNT(*) as orders_count,
    SUM(total_amount) as total_revenue,
    AVG(total_amount) as avg_order_value,
    COUNT(DISTINCT customer_id) as unique_customers
FROM orders
GROUP BY DATE_TRUNC('month', order_date::DATE)
ORDER BY month;
"""

monthly_sales = con.execute(monthly_sales_query).fetchdf()
print("Monthly sales analysis:")
monthly_sales

Monthly sales analysis:


,month,orders_count,total_revenue,avg_order_value,unique_customers
0,2023-12-01,29,2185.48,75.361379,29
1,2024-01-01,104,9564.58,91.967115,101
2,2024-02-01,95,8296.71,87.333789,90
3,2024-03-01,94,10445.01,111.117128,90
4,2024-04-01,107,11034.70,103.128037,105
5,2024-05-01,120,9788.77,81.573083,109
6,2024-06-01,121,12317.52,101.797686,117
7,2024-07-01,105,9796.58,93.300762,101
8,2024-08-01,119,9450.62,79.416975,112
9,2024-09-01,101,10145.43,100.449802,95


In [12]:
# Product performance analysis
# Orders connect to products through the order_items table
product_performance_query = """
SELECT
    p.product_id,
    p.product_name,
    p.price,
    COUNT(oi.order_item_id) as times_ordered,
    SUM(oi.quantity) as units_sold,
    SUM(oi.total_price) as total_revenue
FROM products p
LEFT JOIN order_items oi ON p.product_id = oi.product_id
GROUP BY p.product_id, p.product_name, p.price
ORDER BY total_revenue DESC NULLS LAST
LIMIT 10;
"""

product_performance = con.execute(product_performance_query).fetchdf()
print("Product performance:")
product_performance

Product performance:


,product_id,product_name,price,times_ordered,units_sold,total_revenue
0,721,Sofa Standard 335,1574.40,7,25.0,38273.69
1,1009,Sofa Sofa 830,1724.93,5,19.0,32421.80
2,861,Laptop Laptop 466,1388.39,7,22.0,30544.58
3,192,Laptop Laptop 461,1310.54,8,21.0,27521.34
4,85,Laptop Laptop 555,1127.61,9,25.0,27079.22
5,855,Smartphone Smartphone 913,977.21,9,27.0,26251.57
6,943,Laptop Standard 273,1220.36,7,21.0,24800.05
7,116,Laptop Laptop 773,1190.24,5,20.0,23804.80
8,986,Sofa Pro 331,1201.57,6,20.0,23318.00
9,1035,Smartphone Smartphone 780,1047.00,7,22.0,22608.77


## Advanced DuckDB Features

DuckDB has many powerful features for analytics.

In [13]:
# Using DuckDB's window functions
window_query = """
SELECT
    customer_id,
    order_date,
    total_amount,
    SUM(total_amount) OVER (PARTITION BY customer_id ORDER BY order_date) as running_total,
    ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date) as order_number
FROM orders
ORDER BY customer_id, order_date
LIMIT 15;
"""

window_results = con.execute(window_query).fetchdf()
print("Window functions example (running totals by customer):")
window_results

Window functions example (running totals by customer):


,customer_id,order_date,total_amount,running_total,order_number
0,1,2024-06-29 22:49:46,75.47,75.47,1
1,1,2024-08-18 22:49:46,54.58,130.05,2
2,2,2024-03-20 22:49:46,88.00,88.00,1
3,2,2024-07-30 22:49:46,68.21,156.21,2
4,3,2025-05-18 22:49:46,171.17,171.17,1
5,4,2025-09-08 22:49:46,71.23,71.23,1
6,5,2024-06-09 22:49:46,59.22,59.22,1
7,5,2025-06-07 22:49:46,0.00,59.22,2
8,7,2025-08-17 22:49:46,56.57,56.57,1
9,7,2025-12-14 22:49:46,80.65,137.22,2


In [14]:
# Using DuckDB's approximate functions for large datasets
approx_query = """
SELECT
    COUNT(*) as total_orders,
    APPROX_COUNT_DISTINCT(customer_id) as unique_customers,
    APPROX_QUANTILE(total_amount, 0.5) as median_order_value,
    APPROX_QUANTILE(total_amount, 0.95) as p95_order_value
FROM orders;
"""

approx_stats = con.execute(approx_query).fetchdf()
print("Approximate statistics:")
approx_stats

Approximate statistics:


,total_orders,unique_customers,median_order_value,p95_order_value
0,2500,1188,74.229092,231.157584


## Practice Exercises

Now it's your turn! Try these exercises to practice your DuckDB skills.

In [15]:
# Exercise 1: Find the top 5 customers by number of orders
exercise1_query = """
-- Write your query here
SELECT customer_id, COUNT(*) as order_count
FROM orders
GROUP BY customer_id
ORDER BY order_count DESC
LIMIT 5;
"""

# Uncomment to run:
# result1 = con.execute(exercise1_query).fetchdf()
# result1

In [ ]:
# Exercise 2: Calculate daily order statistics for the last 30 days of data
exercise2_query = """
-- Write your query here
WITH max_date AS (SELECT MAX(order_date::DATE) as m FROM orders)
SELECT
    DATE_TRUNC('day', order_date::DATE) as order_day,
    COUNT(*) as daily_orders,
    SUM(total_amount) as daily_revenue
FROM orders, max_date
WHERE order_date::DATE >= max_date.m - INTERVAL 30 DAY
GROUP BY DATE_TRUNC('day', order_date::DATE)
ORDER BY order_day;
"""

# Uncomment to run:
# result2 = con.execute(exercise2_query).fetchdf()
# result2

In [ ]:
# Exercise 3: Find customers who haven't ordered in the last 90 days (relative to the most recent order)
exercise3_query = """
-- Write your query here
WITH last_order_dates AS (
    SELECT
        customer_id,
        MAX(order_date::DATE) as last_order_date
    FROM orders
    GROUP BY customer_id
),
reference_date AS (SELECT MAX(order_date::DATE) as ref FROM orders)
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    c.email,
    l.last_order_date,
    r.ref - l.last_order_date as days_since_last_order
FROM customers c
JOIN last_order_dates l ON c.customer_id = l.customer_id
CROSS JOIN reference_date r
WHERE l.last_order_date < r.ref - INTERVAL 90 DAY
ORDER BY days_since_last_order DESC;
"""

# Uncomment to run:
# result3 = con.execute(exercise3_query).fetchdf()
# result3

## Working with Polars Integration

DuckDB integrates seamlessly with Polars for fast data analysis workflows.

In [ ]:
# Read data directly into Polars
customers_pl = con.execute("SELECT * FROM customers").pl()
orders_pl = con.execute("SELECT * FROM orders").pl()

print("Customers shape:", customers_pl.shape)
print("Orders shape:", orders_pl.shape)

# Perform analysis with Polars
merged_df = orders_pl.join(customers_pl, on="customer_id", how="left")
print("\nMerged data sample:")
print(merged_df.head())

In [ ]:
# Use DuckDB's Polars integration for complex queries
complex_query = """
SELECT
    c.first_name,
    c.last_name,
    COUNT(o.order_id) as order_count,
    SUM(o.total_amount) as total_spent,
    AVG(o.total_amount) as avg_order_value
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.first_name, c.last_name
HAVING COUNT(o.order_id) > 0
ORDER BY total_spent DESC
LIMIT 10;
"""

top_customers = con.execute(complex_query).pl()
print("Top customers analysis:")
print(top_customers)

## Closing the Connection

Always remember to close your database connection when done.

In [ ]:
# Close the connection
con.close()
print("Database connection closed.")

## Key Takeaways

- DuckDB is perfect for analytical workloads and local development
- It integrates seamlessly with Python and Polars
- Standard SQL syntax with powerful extensions
- Great for prototyping before moving to production databases
- Excellent performance for complex analytical queries

## Next Steps

- Explore DuckDB's documentation: https://duckdb.org/
- Try the exercises above
- Experiment with different file formats (CSV, Parquet)
- Learn about DuckDB's advanced features like extensions
- Check out Polars documentation: https://pola.rs/